# AIC 2026 — Retry summary cho các video bị FAILED (Qwen3.5-4B, 2 GPU Kaggle)

Notebook **riêng, gọn**, chỉ để chạy lại đúng những video đã fail ở lượt trước (không đụng gì tới 415/458
video đã chạy xong, không có logic chọn folder/resume rối rắm).

**Vì sao fail**: lỗi gốc là `RuntimeError('Already borrowed')` — 2 GPU thread cùng gọi chung một
`tokenizer` HuggingFace (fast, viết bằng Rust, không thread-safe) đồng thời. Notebook này đã sửa: mọi lệnh
gọi vào tokenizer đều đi qua `TOKENIZER_LOCK` nên lỗi này không lặp lại.

**Cách dùng**:
1. Sửa `FAILED_VIDEO_IDS` ở cell **Cấu hình** nếu danh sách video fail khác (đọc từ `_failed.json` của lượt trước).
2. Kaggle → Settings → Accelerator **GPU T4 x2**, Internet **On**.
3. **Add Data**: dataset caption (`Image_captioning`) và dataset transcript (`Transcript_Extract`) — giống notebook gốc.
4. Run All. Output nằm ở `/kaggle/working/Summary_video/<video_id>.json` + `.txt`, và một file
   `summaries_retry.zip` gộp tất cả để tải về.
5. Tải `summaries_retry.zip` (tab **Output** bên phải), giải nén, copy các file vào
   `Feature_Dataset/Summary_video/` trong repo (đè lên bản cũ nếu có).

In [ ]:
import os
import subprocess
import sys

def detect_env():
    """Phát hiện môi trường: 'kaggle' | 'colab' | 'local'."""
    if os.path.isdir('/kaggle/input') or os.path.isdir('/kaggle/working') or os.environ.get('KAGGLE_KERNEL_RUN_TYPE'):
        return 'kaggle'
    if os.environ.get('COLAB_RELEASE_TAG') or 'google.colab' in sys.modules or (os.name == 'posix' and os.path.isdir('/content')):
        return 'colab'
    return 'local'

ENV = detect_env()
print('Môi trường:', ENV)

subprocess.run(['nvidia-smi'], check=False)

# Qwen3.5 cần transformers mới; KHÔNG đụng torch/numpy của Kaggle/Colab.
subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', '-U', 'transformers', 'accelerate'], check=True)

import transformers
print('transformers:', transformers.__version__)

In [ ]:
if ENV == 'colab':
    from google.colab import drive
    drive.mount('/content/drive')
else:
    print('Bỏ qua mount Drive (ENV =', ENV, ')')

## Cấu hình

`FAILED_VIDEO_IDS` là danh sách video cần chạy lại — copy từ `_failed.json` của lượt trước (mặc định đã
điền sẵn 8 video bị lỗi `RuntimeError('Already borrowed')`). Notebook chỉ dựng timeline và chạy đúng những
video này, không quét/chọn cả folder.

Đường dẫn caption/transcript dò tự động theo tên thư mục trong `/kaggle/input`, giống notebook gốc; điền
tay vào `CAPTION_DIRS_OVERRIDE` / `TRANSCRIPT_DIRS_OVERRIDE` nếu dò sai.

In [ ]:
from pathlib import Path

try:
    ENV
except NameError:
    ENV = ('kaggle' if os.path.isdir('/kaggle/input')
           else 'colab' if (os.name == 'posix' and os.path.isdir('/content')) else 'local')

# --- Video cần chạy lại -----------------------------------------------------
FAILED_VIDEO_IDS = [
    'L26_V072',
    'L26_V073',
    'L26_V074',
    'L26_V075',
    'L26_V076',
    'L26_V077',
    'L26_V078',
    'L26_V079',
]

# --- Tên thư mục nguồn (dò theo tên, không hard-code cả đường dẫn) ----------
CAPTION_DIR_NAMES = ['Image_captioning', 'ImageCaptioning', 'Captions', 'VLM_Qwen3.5-2b']
TRANSCRIPT_DIR_NAMES = ['Transcript_Extract', 'Transcripts']

CAPTION_DIRS_OVERRIDE = []      # ví dụ: [Path('/kaggle/input/datasets/<user>/<ds>/Image_captioning')]
TRANSCRIPT_DIRS_OVERRIDE = [
    Path('/kaggle/input/datasets/kitnehi1211/transcript/Transcript_Extract'),
]

if ENV == 'kaggle':
    SEARCH_ROOTS = [Path('/kaggle/input'), Path('/kaggle/working')]
    OUTPUT_ROOT = Path('/kaggle/working/Summary_video')
elif ENV == 'colab':
    SEARCH_ROOTS = [Path('/content/drive/MyDrive/AI Challenge')]
    OUTPUT_ROOT = Path('/content/drive/MyDrive/AI Challenge/Summary_video')
else:
    SEARCH_ROOTS = [Path('./Feature_Dataset'), Path('../Feature_Dataset'), Path('../../Feature_Dataset')]
    OUTPUT_ROOT = Path('./Feature_Dataset/Summary_video')

def find_dirs(names, roots, max_depth=4):
    wanted = {name.lower() for name in names}
    found = []
    for root in roots:
        if not root.is_dir():
            continue
        for depth in range(1, max_depth + 1):
            for path in root.glob('/'.join(['*'] * depth)):
                if path.is_dir() and path.name.lower() in wanted:
                    found.append(path.resolve())
    output_resolved = OUTPUT_ROOT.resolve()
    unique = []
    for path in found:
        if path == output_resolved or output_resolved in path.parents:
            continue
        if path not in unique:
            unique.append(path)
    return unique

def resolve_sources(overrides, names):
    chosen = [Path(path).resolve() for path in overrides if Path(path).is_dir()]
    missing = [str(path) for path in overrides if not Path(path).is_dir()]
    for path in missing:
        print('  (override không tồn tại, bỏ qua):', path)
    return chosen or find_dirs(names, SEARCH_ROOTS)

CAPTION_DIRS = resolve_sources(CAPTION_DIRS_OVERRIDE, CAPTION_DIR_NAMES)
TRANSCRIPT_DIRS = resolve_sources(TRANSCRIPT_DIRS_OVERRIDE, TRANSCRIPT_DIR_NAMES)

print('Thư mục caption:')
for path in CAPTION_DIRS or ['(không tìm thấy)']:
    print('  -', path)
print('Thư mục transcript:')
for path in TRANSCRIPT_DIRS or ['(không tìm thấy)']:
    print('  -', path)
if not CAPTION_DIRS and not TRANSCRIPT_DIRS and ENV == 'kaggle':
    print('\nCó trong /kaggle/input:', ', '.join(sorted(p.name for p in Path('/kaggle/input').iterdir())))
assert CAPTION_DIRS or TRANSCRIPT_DIRS, (
    'Không tìm thấy nguồn nào. Add Data dataset caption/transcript rồi điền CAPTION_DIRS_OVERRIDE.'
)

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
OUTPUT_ROOT = OUTPUT_ROOT.resolve()
print('Output:', OUTPUT_ROOT)

OVERWRITE = True   # chạy lại là để đè lên kết quả fail cũ (nếu có), luôn ghi mới

# --- Model -----------------------------------------------------------------
MODEL_ID = 'Qwen/Qwen3.5-4B'
DTYPE = 'auto'
ENABLE_THINKING = False

# --- Chunk + sinh text -----------------------------------------------------
CHUNK_TOKEN_BUDGET = 2600
MAP_BATCH_SIZE = 4
MAX_CHUNKS_PER_VIDEO = 0
MAX_NEW_TOKENS_MAP = 320
MAX_NEW_TOKENS_FINAL = 420

DO_SAMPLE = True
TEMPERATURE = 0.7
TOP_P = 0.8
TOP_K = 20
SEED = 1234

# --- Nội dung nguồn --------------------------------------------------------
SKIP_DUPLICATE_CAPTIONS = True
SPEECH_LANGUAGE = 'Vietnamese'

## Prompt

Giống nguyên bản: `SUMMARY` / `TOPICS` / `ENTITIES` được parse bằng regex ở cell sau.

In [ ]:
SYSTEM_PROMPT = (
    'You are an expert video analyst building English metadata for a video retrieval system. '
    'You never see the video itself; you receive an evidence timeline extracted from it: '
    f'VISUAL lines are English captions of keyframes, SPEECH lines are {SPEECH_LANGUAGE} '
    'automatic transcripts. Translate any non-English content into English. '
    'Ground every statement in the evidence; never invent names, numbers or places. '
    'Automatic transcripts are noisy - ignore fragments that make no sense instead of guessing.'
)

MAP_PROMPT = '''Below is part of the evidence timeline of video {video_id} (part {part}/{total}, {start}-{end}).

{evidence}

Write a dense factual English digest of this part, as 3-6 bullet lines starting with "- ".
Each bullet: what is on screen and what is being said, with the approximate timestamp in [mm:ss].
Keep every concrete detail: people and their roles, clothing, objects, counts, locations, organisations,
on-screen text, actions and events. No preamble, no conclusion, bullets only.'''

FINAL_PROMPT = '''Below is the evidence about video {video_id}.

{evidence}

Write English retrieval metadata for this video, in exactly this format:

SUMMARY: one paragraph of 120-200 words describing what happens in the video in chronological order.
State the subject matter, setting, the people involved and the main events. Write plain declarative
sentences about the content itself - do not start with "The video shows" or similar meta phrasing.
TOPICS: 5-10 short topical keywords or phrases, separated by semicolons.
ENTITIES: the named people, organisations, places, dates and numbers that appear, separated by
semicolons; write "none" if the evidence contains none.'''

## Dựng timeline cho các video FAILED

Chỉ đọc caption + transcript của đúng các `video_id` trong `FAILED_VIDEO_IDS`, không quét cả dataset.

In [ ]:
import json
import re

VIDEO_ID_PATTERN = re.compile(r'^L\d{2}_V\d{3}$')

def index_sources(dirs):
    mapping = {}
    for directory in dirs:
        for path in sorted(directory.rglob('*.json')):
            if path.name.startswith('_') or path.stem.endswith('.partial'):
                continue
            if VIDEO_ID_PATTERN.match(path.stem):
                mapping.setdefault(path.stem, path)
    return mapping

CAPTION_FILES = index_sources(CAPTION_DIRS)
TRANSCRIPT_FILES = index_sources(TRANSCRIPT_DIRS)
print(f'Caption: {len(CAPTION_FILES)} video | Transcript: {len(TRANSCRIPT_FILES)} video (toàn dataset)')

def read_json(path):
    try:
        return json.loads(path.read_text(encoding='utf-8'))
    except Exception as error:
        print(f'Bỏ qua file lỗi {path}: {error!r}')
        return None

def caption_events(video_id):
    path = CAPTION_FILES.get(video_id)
    payload = read_json(path) if path else None
    if not payload:
        return []
    events, previous = [], None
    for item in payload.get('keyframes', []):
        if SKIP_DUPLICATE_CAPTIONS and item.get('duplicate_of'):
            continue
        text = (item.get('caption') or '').strip()
        if not text:
            continue
        normalised = re.sub(r'\W+', ' ', text.lower()).strip()
        if normalised == previous:
            continue
        previous = normalised
        try:
            time = float(item.get('pts_time') or 0.0)
        except (TypeError, ValueError):
            time = 0.0
        events.append({'t': time, 'kind': 'VISUAL', 'text': text})
    return events

def speech_events(video_id):
    path = TRANSCRIPT_FILES.get(video_id)
    payload = read_json(path) if path else None
    if not payload:
        return []
    events = []
    for segment in payload.get('segments', []):
        text = (segment.get('text') or '').strip()
        if not text:
            continue
        try:
            time = float(segment.get('video_start') if segment.get('video_start') is not None else segment.get('start') or 0.0)
        except (TypeError, ValueError):
            time = 0.0
        events.append({'t': time, 'kind': 'SPEECH', 'text': text})
    if not events:
        whole = (payload.get('text') or '').strip()
        if whole:
            events.append({'t': 0.0, 'kind': 'SPEECH', 'text': whole})
    return events

def clock(seconds):
    total = max(0, int(round(float(seconds))))
    return f'{total // 60:02d}:{total % 60:02d}'

def build_timeline(video_id):
    visual, speech = caption_events(video_id), speech_events(video_id)
    events = sorted(visual + speech, key=lambda item: (item['t'], item['kind']))
    return {
        'video_id': video_id,
        'events': events,
        'visual_count': len(visual),
        'speech_count': len(speech),
        'has_caption': video_id in CAPTION_FILES,
        'has_transcript': video_id in TRANSCRIPT_FILES,
        'duration_hint': max((item['t'] for item in events), default=0.0),
    }

def event_line(event):
    return f"[{clock(event['t'])}] {event['kind']}: {event['text']}"

missing = [video_id for video_id in FAILED_VIDEO_IDS
           if video_id not in CAPTION_FILES and video_id not in TRANSCRIPT_FILES]
if missing:
    print('CẢNH BÁO: không tìm thấy caption/transcript cho:', missing)

timelines = {}
for video_id in FAILED_VIDEO_IDS:
    timeline = build_timeline(video_id)
    if timeline['events']:
        timelines[video_id] = timeline
    else:
        print(f'  {video_id}: không có sự kiện nào (rỗng) -> bỏ qua')

video_ids = list(timelines)
print(f'\n{len(video_ids)}/{len(FAILED_VIDEO_IDS)} video có dữ liệu, sẽ chạy:')
for video_id in video_ids:
    timeline = timelines[video_id]
    print(f"  - {video_id}: {timeline['visual_count']} caption, {timeline['speech_count']} câu nói, "
          f"~{clock(timeline['duration_hint'])}")

## Tải model lên 2 GPU

Giống nguyên bản: mỗi GPU giữ một bản model riêng. `TOKENIZER_LOCK` bọc quanh mọi lệnh gọi vào
`tokenizer` (dùng chung giữa 2 thread) — đây là fix cho lỗi `RuntimeError('Already borrowed')` đã làm
8 video này fail ở lượt trước.

In [ ]:
import torch
from threading import Lock
from transformers import AutoModelForCausalLM, AutoTokenizer

assert torch.cuda.is_available(), (
    'Chưa bật GPU: Kaggle → Settings → Accelerator → GPU T4 x2'
    if ENV == 'kaggle' else 'Chưa bật GPU: Runtime → Change runtime type → GPU'
)

if DTYPE == 'auto':
    model_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
else:
    model_dtype = {'float16': torch.float16, 'bfloat16': torch.bfloat16}[DTYPE]

GPU_COUNT = min(2, torch.cuda.device_count(), max(1, len(video_ids)))
print(f'Phát hiện {torch.cuda.device_count()} GPU; sẽ dùng {GPU_COUNT} | dtype: {model_dtype}')
for gpu_id in range(GPU_COUNT):
    print(f'  cuda:{gpu_id}: {torch.cuda.get_device_name(gpu_id)}')

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.padding_side = 'left'
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Tokenizer "fast" (Rust) KHÔNG thread-safe -> mọi lệnh gọi vào nó phải qua lock này.
TOKENIZER_LOCK = Lock()

def build_model(gpu_id):
    print(f'[GPU {gpu_id}] Đang tải {MODEL_ID}...')
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, dtype=model_dtype, low_cpu_mem_usage=True, attn_implementation='sdpa',
        trust_remote_code=True,
    ).to(f'cuda:{gpu_id}')
    model.eval()
    print(f'[GPU {gpu_id}] Đã tải model')
    return model

models = [build_model(gpu_id) for gpu_id in range(GPU_COUNT)]
RUN_ID = f'{MODEL_ID}|summary-en|think={int(bool(ENABLE_THINKING))}|v1'
print('RUN_ID:', RUN_ID)

## Sinh summary

Y hệt logic nguyên bản (map-reduce theo chunk), chỉ khác là mọi lệnh gọi tokenizer đi qua `TOKENIZER_LOCK`.

In [ ]:
import traceback

def count_tokens(text):
    with TOKENIZER_LOCK:
        return len(tokenizer(text, add_special_tokens=False).input_ids)

def chunk_events(events, budget=None):
    budget = budget or CHUNK_TOKEN_BUDGET
    chunks, current, current_tokens = [], [], 0
    for event in events:
        line = event_line(event)
        tokens = count_tokens(line) + 1
        if current and current_tokens + tokens > budget:
            chunks.append(current)
            current, current_tokens = [], 0
        if tokens > budget:
            with TOKENIZER_LOCK:
                ids = tokenizer(line, add_special_tokens=False).input_ids[:budget]
                line = tokenizer.decode(ids)
            tokens = budget
        current.append(line)
        current_tokens += tokens
    if current:
        chunks.append(current)
    if MAX_CHUNKS_PER_VIDEO:
        chunks = chunks[:MAX_CHUNKS_PER_VIDEO]
    return chunks

def render_prompt(user_text):
    messages = [{'role': 'system', 'content': SYSTEM_PROMPT}, {'role': 'user', 'content': user_text}]
    kwargs = {'tokenize': False, 'add_generation_prompt': True}
    with TOKENIZER_LOCK:
        try:
            return tokenizer.apply_chat_template(messages, enable_thinking=ENABLE_THINKING, **kwargs)
        except TypeError:
            return tokenizer.apply_chat_template(messages, **kwargs)

THINK_BLOCK = re.compile(r'<think>.*?</think>', re.DOTALL)

def clean_output(text):
    text = THINK_BLOCK.sub('', text)
    text = text.replace('<think>', '').replace('</think>', '')
    return text.strip()

def generate_batch(model, prompts, max_new_tokens):
    if not prompts:
        return []
    device = next(model.parameters()).device
    try:
        with TOKENIZER_LOCK:
            batch = tokenizer(prompts, return_tensors='pt', padding=True, add_special_tokens=False).to(device)
        torch.manual_seed(SEED)
        with torch.inference_mode():
            output = model.generate(
                **batch,
                max_new_tokens=max_new_tokens,
                do_sample=DO_SAMPLE,
                temperature=TEMPERATURE if DO_SAMPLE else None,
                top_p=TOP_P if DO_SAMPLE else None,
                top_k=TOP_K if DO_SAMPLE else None,
                pad_token_id=tokenizer.pad_token_id,
            )
        generated = output[:, batch.input_ids.shape[1]:]
        with TOKENIZER_LOCK:
            decoded = tokenizer.batch_decode(generated, skip_special_tokens=True)
        return [clean_output(text) for text in decoded]
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        if len(prompts) == 1:
            raise
        print(f'  OOM ở batch {len(prompts)} -> lùi về từng prompt một')
        results = []
        for prompt in prompts:
            results.extend(generate_batch(model, [prompt], max_new_tokens))
        return results

SECTION_PATTERN = re.compile(
    r'^\s*(SUMMARY|TOPICS|ENTITIES)\s*[:\-]\s*(.*?)(?=^\s*(?:SUMMARY|TOPICS|ENTITIES)\s*[:\-]|\Z)',
    re.IGNORECASE | re.DOTALL | re.MULTILINE,
)

def split_list(text):
    parts = [part.strip(' .;-') for part in re.split(r'[;\n]|(?:^|\s)[-*]\s', text) if part.strip(' .;-')]
    if len(parts) <= 1 and ',' in text:
        parts = [part.strip() for part in text.split(',') if part.strip()]
    return [part for part in parts if part.lower() not in {'none', 'n/a'}]

def parse_final(text):
    sections = {name.upper(): body.strip() for name, body in SECTION_PATTERN.findall(text)}
    summary = sections.get('SUMMARY', '').strip()
    if not summary:
        summary = re.sub(r'^\s*(TOPICS|ENTITIES)\s*[:\-].*$', '', text, flags=re.IGNORECASE | re.MULTILINE).strip()
    return {
        'summary': ' '.join(summary.split()),
        'topics': split_list(sections.get('TOPICS', '')),
        'entities': split_list(sections.get('ENTITIES', '')),
    }

def summarize_timeline(model, timeline, log=print):
    video_id, events = timeline['video_id'], timeline['events']
    chunks = chunk_events(events)
    chunk_summaries = []

    if len(chunks) > 1:
        prompts, spans = [], []
        for index, lines in enumerate(chunks, 1):
            start, end = lines[0].split(']')[0].lstrip('['), lines[-1].split(']')[0].lstrip('[')
            spans.append((start, end))
            prompts.append(render_prompt(MAP_PROMPT.format(
                video_id=video_id, part=index, total=len(chunks),
                start=start, end=end, evidence='\n'.join(lines),
            )))
        log(f'  map: {len(chunks)} chunk')
        for offset in range(0, len(prompts), MAP_BATCH_SIZE):
            chunk_summaries.extend(generate_batch(model, prompts[offset:offset + MAP_BATCH_SIZE], MAX_NEW_TOKENS_MAP))
        evidence = '\n\n'.join(
            f'--- part {index}/{len(chunks)} ({spans[index - 1][0]}-{spans[index - 1][1]}) ---\n{summary}'
            for index, summary in enumerate(chunk_summaries, 1)
        )
    else:
        evidence = '\n'.join(chunks[0]) if chunks else ''

    final_text = generate_batch(
        model, [render_prompt(FINAL_PROMPT.format(video_id=video_id, evidence=evidence))], MAX_NEW_TOKENS_FINAL,
    )[0]
    parsed = parse_final(final_text)
    return {
        'video_id': video_id,
        'model': MODEL_ID,
        'run_id': RUN_ID,
        'language': 'en',
        'enable_thinking': bool(ENABLE_THINKING),
        'summary': parsed['summary'],
        'topics': parsed['topics'],
        'entities': parsed['entities'],
        'raw_output': final_text,
        'chunk_summaries': chunk_summaries,
        'num_chunks': len(chunks),
        'evidence': {
            'visual_count': timeline['visual_count'],
            'speech_count': timeline['speech_count'],
            'has_caption': timeline['has_caption'],
            'has_transcript': timeline['has_transcript'],
            'duration_hint': round(timeline['duration_hint'], 2),
            'caption_source': str(CAPTION_FILES.get(video_id, '')),
            'transcript_source': str(TRANSCRIPT_FILES.get(video_id, '')),
        },
        'complete': True,
    }

def output_json_path(video_id):
    return OUTPUT_ROOT / f'{video_id}.json'

def atomic_write(path, text):
    temp = path.with_suffix(path.suffix + '.tmp')
    temp.write_text(text, encoding='utf-8')
    temp.replace(path)

def save_summary(payload):
    video_id = payload['video_id']
    atomic_write(output_json_path(video_id), json.dumps(payload, ensure_ascii=False, indent=2))
    lines = [payload['summary'], '']
    if payload['topics']:
        lines.append('TOPICS: ' + '; '.join(payload['topics']))
    if payload['entities']:
        lines.append('ENTITIES: ' + '; '.join(payload['entities']))
    atomic_write(OUTPUT_ROOT / f'{video_id}.txt', '\n'.join(lines) + '\n')

## Chạy lại các video FAILED

Chỉ đúng số video trong `video_ids` (đã lọc từ `FAILED_VIDEO_IDS`), chia đều cho GPU qua một queue chung.
Video nào vẫn lỗi được ghi vào `_failed_retry.json` kèm traceback đầy đủ để soi tiếp.

In [ ]:
from concurrent.futures import ThreadPoolExecutor
from queue import Empty, Queue
from threading import Lock as PrintLockCls
import time

jobs = Queue()
for index, video_id in enumerate(video_ids, 1):
    jobs.put((index, video_id))
total = len(video_ids)
print_lock = PrintLockCls()

def gpu_worker(gpu_id):
    model = models[gpu_id]
    success = failed = 0
    failures = []
    with torch.cuda.device(gpu_id):
        while True:
            try:
                index, video_id = jobs.get_nowait()
            except Empty:
                break
            try:
                with print_lock:
                    print(f'[GPU {gpu_id}] [{index}/{total}] SUM  {video_id}')
                started = time.time()
                payload = summarize_timeline(model, timelines[video_id], log=lambda message: None)
                save_summary(payload)
                success += 1
                with print_lock:
                    print(f'[GPU {gpu_id}] [{index}/{total}] OK   {video_id} '
                          f'({time.time() - started:.0f}s, {payload["num_chunks"]} chunk, '
                          f'{len(payload["summary"].split())} từ)')
            except Exception as error:
                failed += 1
                failures.append({'video_id': video_id, 'gpu': gpu_id, 'error': repr(error)})
                with print_lock:
                    print(f'[GPU {gpu_id}] ERROR {video_id}: {error!r}')
                    print(traceback.format_exc())
            finally:
                jobs.task_done()
                torch.cuda.empty_cache()
    return success, failed, failures

with ThreadPoolExecutor(max_workers=GPU_COUNT) as executor:
    worker_results = list(executor.map(gpu_worker, range(GPU_COUNT)))

success = sum(item[0] for item in worker_results)
failed = sum(item[1] for item in worker_results)
failures = [failure for item in worker_results for failure in item[2]]
if failures:
    atomic_write(OUTPUT_ROOT / '_failed_retry.json', json.dumps(failures, ensure_ascii=False, indent=2))
print(f'\nHoàn tất: success={success}, failed={failed}, total={total}')
if failures:
    print('Vẫn còn fail (xem _failed_retry.json):', [f['video_id'] for f in failures])
print('Output:', OUTPUT_ROOT)

## Kiểm tra nhanh + đóng gói để tải về

In summary của các video vừa chạy để soi bằng mắt, rồi nén thành `summaries_retry.zip` — tải file này ở
tab **Output**, giải nén, copy `<video_id>.json` + `<video_id>.txt` vào `Feature_Dataset/Summary_video/`
trong repo (đè lên nếu đã có bản lỗi/rỗng cũ).

In [ ]:
import shutil
import textwrap

for video_id in video_ids:
    payload = read_json(output_json_path(video_id)) or {}
    print(f"=== {payload.get('video_id', video_id)} ({payload.get('num_chunks', '?')} chunk) ===")
    print(textwrap.fill(payload.get('summary', '') or '(KHÔNG CÓ - xem log lỗi ở trên)', 100))
    print('TOPICS:', '; '.join(payload.get('topics') or []))
    print('ENTITIES:', '; '.join(payload.get('entities') or []), '\n')

empty = [video_id for video_id in video_ids if not (read_json(output_json_path(video_id)) or {}).get('summary')]
if empty:
    print('CẢNH BÁO: summary rỗng/thiếu ở:', ', '.join(empty))

if ENV == 'kaggle':
    archive = shutil.make_archive('/kaggle/working/summaries_retry', 'zip', root_dir=OUTPUT_ROOT)
    print(f'Đã đóng gói: {archive} ({Path(archive).stat().st_size / 1024:.0f} KB)')
    print('Tải về ở panel Output bên phải, giải nén rồi copy các file .json/.txt vào Feature_Dataset/Summary_video/.')
else:
    print('Bỏ qua đóng gói: kết quả đã nằm ở', OUTPUT_ROOT)